In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from loguru import logger

# ==================== サブグラフ ====================
class SubState(TypedDict):
    data: str

def sub_node(state: SubState) -> Command:
    """サブグラフのノード：実行後に親グラフの parent_router へ動的にルーティングする"""
    logger.info("[サブグラフ] sub_node を実行")
    return Command(
        update={"data": state["data"] + " → サブグラフ"},
        goto="node_b",       # 親グラフのノードへルーティング
        graph=Command.PARENT,       # ターゲットを親グラフに指定
    )

sub_builder = StateGraph(state_schema=SubState)
sub_builder.add_node("sub_node", sub_node)
sub_builder.add_edge(START, "sub_node")
sub_graph = sub_builder.compile()

# ==================== 親グラフ ====================
class ParentState(TypedDict):
    data: str

def node_a(state: ParentState) -> ParentState:
    logger.info("[親グラフ] node_a を実行")
    return {"visited_a": True}

def node_b(state: ParentState) -> ParentState:
    logger.info("[親グラフ] node_b を実行")
    return {"visited_b": True}

parent_builder = StateGraph(state_schema=ParentState)
parent_builder.add_node(
    "sub_graph", sub_graph,destinations=("node_a","node_b")
)

parent_builder.add_node("node_a", node_a)
parent_builder.add_node("node_b", node_b)
parent_builder.add_edge(START, "sub_graph")
# parent_builder.add_edge("sub_graph","node_a")
parent_builder.add_edge("node_a", "node_b")
parent_builder.add_edge("node_b", END)

parent_graph = parent_builder.compile()

result = parent_graph.invoke({"data": "初期状態"})
print(f"\n最終結果: {result}")

from IPython.display import display
display(parent_graph)